# AssureX Claim Engine — Teachable Machine Image Classifier

**Project:** AssureX Claim Engine  
**SRS Reference:** Sections 1.2 Steps 7–9 (Card Generation, GTM Training, GTM Classification)

This notebook covers:
1. Claim Summary Card image generation from the dataset
2. Training an image classification model on card images (equivalent to Google Teachable Machine)
3. Evaluating the image model on validation and test sets
4. Comparing Python ML model vs. Image model predictions
5. Confidence score comparison and model consistency analysis

**SRS Note:** The image-based model trained here is functionally equivalent to Google Teachable Machine.  
Both models use the same 1500 underlying claim records in different representations.


## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import json, os, csv
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from PIL import Image

BASE_DIR  = Path('..')
CARD_DIR  = BASE_DIR / 'data' / 'claim_cards'
MODEL_DIR = BASE_DIR / 'model' / 'teachable_machine'
PROC_DIR  = BASE_DIR / 'data' / 'processed'

CLASSES = ['invalid_claim', 'manual_review', 'valid_claim']

print("Setup complete.")
print(f"  Card directory   : {CARD_DIR}")
print(f"  TM model directory: {MODEL_DIR}")


## 2. Claim Summary Card Dataset Overview

In [ ]:
print("Claim Summary Card Image Counts:")
print("-" * 45)
grand_total = 0
for split in ['train', 'validation', 'test']:
    total = 0
    for cls in CLASSES:
        count = len(list((CARD_DIR / split / cls).glob('*.png')))
        print(f"  {split:<12} / {cls:<18}: {count:>4} images")
        total += count
    print(f"  {split:<12}   {'TOTAL':<18}: {total:>4} images")
    grand_total += total
    print()
print(f"Grand total: {grand_total} images")
print(f"SRS requirement (≥2100 training): {'MET ✓' if grand_total >= 2100 else 'NOT MET ✗'}")


In [ ]:
# Show sample cards from each class
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for row, cls in enumerate(CLASSES):
    cls_dir = CARD_DIR / 'train' / cls
    imgs = sorted(cls_dir.glob('*.png'))[:3]
    for col, img_path in enumerate(imgs):
        img = Image.open(img_path)
        axes[row][col].imshow(img)
        axes[row][col].set_title(f'{cls.replace("_", " ").title()}\nVar {col}', fontsize=9)
        axes[row][col].axis('off')

plt.suptitle('Sample Claim Summary Cards (3 classes × 3 variations)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/tm_sample_cards.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved → reports/tm_sample_cards.png")


## 3. Teachable Machine Model — Training Details

In [ ]:
meta_path = MODEL_DIR / 'metadata.json'
if meta_path.exists():
    with open(meta_path, encoding='utf-8') as f:
        meta = json.load(f)

    print("Teachable Machine Image Classifier — Training Metadata")
    print("=" * 55)
    print(f"  Architecture  : {meta['architecture']}")
    print(f"  Image size    : {meta['image_input_size']}")
    print(f"  Feature count : {meta['feature_count']}")
    print(f"  Train images  : {meta['train_images']}")
    print(f"  Val images    : {meta['val_images']}")
    print(f"  Test images   : {meta['test_images']}")
    print(f"  Test accuracy : {meta['test_accuracy']:.2f}%")
    print(f"  SRS 85% target: {'MET ✓' if meta['srs_target_85pct'] else 'NOT MET ✗'}")
    print(f"  Version       : {meta['version']}")
    print(f"  Trained at    : {meta['trained_at']}")


In [ ]:
# Load and display training report
report_path = MODEL_DIR / 'tm_training_report.txt'
if report_path.exists():
    print(report_path.read_text(encoding='utf-8'))


## 4. Model Predictions on Test Set

In [ ]:
from src.tm_trainer.sklearn_tm_predictor import predict as tm_predict

MODEL_PKL = str(MODEL_DIR / 'sklearn_tm_model.pkl')

test_records = []
for cls in CLASSES:
    cls_dir = CARD_DIR / 'test' / cls
    for img_path in sorted(cls_dir.glob('*.png')):
        result = tm_predict(str(img_path), MODEL_PKL)
        test_records.append({
            'actual':            cls,
            'predicted':         result['predicted_class'],
            'conf_invalid':      result['confidence_invalid'],
            'conf_manual':       result['confidence_manual'],
            'conf_valid':        result['confidence_valid'],
            'top_confidence':    result['top_confidence'],
            'correct':           cls == result['predicted_class'],
        })

results_df = pd.DataFrame(test_records)
accuracy   = results_df['correct'].mean() * 100
print(f"TM Model Test Accuracy: {accuracy:.2f}%")
print(f"Total test images     : {len(results_df)}")
print(f"Correct predictions   : {results_df['correct'].sum()}")
print(f"Incorrect predictions : {(~results_df['correct']).sum()}")


## 5. Confusion Matrix — TM Image Model

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

y_true = results_df['actual']
y_pred = results_df['predicted']

cm = confusion_matrix(y_true, y_pred, labels=CLASSES)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Invalid', 'Manual', 'Valid'])
fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title(f'TM Image Classifier — Test Set Confusion Matrix\nAccuracy: {accuracy:.2f}%',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/tm_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → reports/tm_confusion_matrix.png")

print()
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=['Invalid', 'Manual', 'Valid'], zero_division=0))


## 6. Confidence Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, cls in zip(axes, CLASSES):
    subset = results_df[results_df['actual'] == cls]['top_confidence']
    ax.hist(subset, bins=20, color='#2563EB', edgecolor='black', alpha=0.8)
    ax.axvline(subset.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean: {subset.mean():.1f}%')
    ax.set_title(f'{cls.replace("_", " ").title()}', fontweight='bold')
    ax.set_xlabel('Top Confidence (%)')
    ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('Confidence Score Distribution by True Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/tm_confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → reports/tm_confidence_distribution.png")


## 7. Incorrectly Classified Samples

In [ ]:
wrong = results_df[~results_df['correct']].reset_index(drop=True)
print(f"Incorrectly classified: {len(wrong)}")
if not wrong.empty:
    print()
    print(wrong[['actual', 'predicted', 'top_confidence']].to_string())


## 8. Summary

In [ ]:
print("=" * 60)
print("AssureX — TM Image Classifier Summary")
print("=" * 60)
print(f"  Training images    : {len(list(CARD_DIR.rglob('*.png'))) - len(results_df)}")
print(f"  Test images        : {len(results_df)}")
print(f"  Test accuracy      : {accuracy:.2f}%")
print(f"  SRS 85% target     : {'MET ✓' if accuracy >= 85 else 'NOT MET ✗'}")
print()
print("  Per-class accuracy:")
for cls in CLASSES:
    sub = results_df[results_df['actual'] == cls]
    acc = sub['correct'].mean() * 100
    print(f"    {cls:<22}: {acc:.2f}%")
print()
print("  Model files at model/teachable_machine/:")
for f in Path('../model/teachable_machine').iterdir():
    print(f"    {f.name}")
print("=" * 60)
